In [0]:
%pip install pysus

In [0]:
from pysus import sih
import os

In [0]:
dbutils.widgets.text("year", "None", "Year (YYYY)")
dbutils.widgets.text("month", "None", "Month (MM)")
dbutils.widgets.text("state", "None", "BR State (GO)")

year = int(dbutils.widgets.get("year"))
month = dbutils.widgets.get("month")
state = dbutils.widgets.get("state")

print(f"Variaveis de entrada: year={year}, month={month}, state={state}")

In [0]:
volume = "/Volumes/oftalmo_sus/00_landing/raw_files"
path = f"{volume}/SIH"
os.makedirs(path, exist_ok=True)

In [0]:
print(f"Iniciando o download diretamente para a Landing: {path}")

In [0]:
import time
from httpx import HTTPStatusError, ReadTimeout

# Configuração de Resiliência
max_tentativas = 5
tentativa_atual = 1
df_sih = None
v_month = int(month)

# Loop de Retry para lidar com o ReadTimeout do DATASUS
while tentativa_atual <= max_tentativas:
    try:
        print(f"⏳ Tentativa {tentativa_atual} de {max_tentativas}...")
        
        # A chamada que pode sofrer timeout
        df_sih = sih(state=state, year=year, month=[v_month])
        
        print("✅ Download do SIH concluído com sucesso do servidor!")
        break # Se der certo, quebra o loop para avançar
        
    except Exception as e:
        print(f"⚠️ Falha na tentativa {tentativa_atual}: O servidor do DATASUS não respondeu a tempo ({e}).")
        
        if tentativa_atual == max_tentativas:
            print("❌ Número máximo de tentativas atingido. Servidor indisponível. Abortando pipeline.")
            raise e # Levanta o erro criticamente para o Databricks alertar falha no Job
            
        print("⏱️ Aguardando 15 segundos antes da próxima tentativa...")
        time.sleep(15)
        tentativa_atual += 1

# 2. Definição do caminho e salvamento (só executa se o loop acima teve sucesso)
if df_sih is not None:
    path_sih = path
    os.makedirs(path_sih, exist_ok=True)
    
    filename = f"{path_sih}/SIH_{state}_{year}{month}.parquet"
    df_sih.to_parquet(filename, index=False)
    
    print(f"📁 Arquivo consolidado e salvo em: {filename}")

In [0]:
landing_fales = os.listdir(path)
print(f"✅ Arquivos atualmente na Landing: {landing_fales}")